# 01c — Files, validation, and money — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/01.md) · [Course map](../PLAN.md) · [Project](../../projects/stage01/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Read CSV and JSON safely
- Validate malformed records
- Aggregate dates and exact decimal values


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


Course root: /Users/sureshkarki/Programming/GitExp/personal/python/learning


## Before the exercises: Files, validation, and exact money

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A path names a file; opening it gives a file object through which data can be read. `with open(...)` closes the file when the block ends, even after an error. CSV is a table format with quoting rules. `csv.DictReader` uses the header to map fields to names. Splitting each line on commas breaks quoted fields containing commas.

Parsing changes text into typed values, such as a date or Decimal amount. Validation checks the values against a contract. Aggregate only validated data, so a partially bad file does not produce an apparently complete report.

Binary floating point approximates many decimal fractions. For money, construct `Decimal('0.10')` from text or use integer cents. Decide explicitly how many decimal places are accepted and what rounding policy applies.

### Worked example: follow the values

Read two rows, amount texts `0.10` and `0.20`. Converting each directly to Decimal gives total `0.30`. Their mean is total divided by count 2. A header-only file has no observations: total can be zero, but mean and maximum have no value and should be represented explicitly as unknown.

### Pause and explain

Why include a row number in a validation error?

<details><summary>Check your reasoning after trying</summary>

The learner or user can locate the bad source record and correct it instead of guessing which value failed.

</details>

### Common mistakes to check

NaN and infinity are numeric-looking but invalid financial amounts. File-not-found is different from an empty file. A command-line program should exit nonzero after an expected input failure.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

A file stores bytes; an encoding maps bytes to text. Use UTF-8 explicitly for portable text. `with open(...)` closes the resource even when an exception occurs. CSV is a table format whose quoting rules belong to a parser, not `split(',')`. JSON stores nested values, but dates and Decimal values need an explicit representation.

Money should not silently accumulate binary floating-point rounding. Decimal constructed from a string preserves the supplied decimal value. Validate at ingestion: required columns, parseable dates, finite nonnegative amounts, and blank categories. Choose a failure policy—reject the whole batch or quarantine bad rows—and explain it. An empty input should have a documented result. Avoid catching all exceptions because that hides defects unrelated to bad data.


## Worked example

Predict the output before running. Change one input and explain the result.


In [2]:
from decimal import Decimal
print(Decimal("0.1") + Decimal("0.2"))
from pathlib import Path
print((ROOT / "datasets" / "expenses.csv").exists())


0.3
True


## Exercise 1: Amount parser

Parse a string as a finite, nonnegative Decimal; raise ValueError on invalid inputs including NaN.

Write down one normal case, one boundary case, and one invalid case before coding.


In [3]:
def parse_amount(value):
    from decimal import Decimal, InvalidOperation
    try:
        amount = Decimal(value)
    except (InvalidOperation, TypeError):
        raise ValueError("Invalid amount") from None
    if not amount.is_finite() or amount < 0:
        raise ValueError("Invalid amount")
    return amount


In [ ]:
from decimal import Decimal
assert parse_amount("12.50") == Decimal("12.50")
for bad in ("NaN", "-1", "oops", "Infinity"):
    expect_error(ValueError, lambda bad=bad: parse_amount(bad))
print("Exercise 1: checks passed")


**Why this works:** NaN and infinity can poison aggregates without obvious parsing failures.


## Exercise 2: CSV summary

For CSV text with date,category,amount, return total Decimal and by_category dict. Validate ISO dates and nonblank categories; use parse_amount.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def summarize_csv(text):
    import csv, io
    from datetime import date
    from decimal import Decimal
    total, groups = Decimal("0"), {}
    for row in csv.DictReader(io.StringIO(text)):
        date.fromisoformat(row["date"])
        category = row["category"].strip()
        if not category: raise ValueError("Blank category")
        amount = parse_amount(row["amount"])
        total += amount
        groups[category] = groups.get(category, Decimal("0")) + amount
    return total, groups


In [ ]:
sample = "date,category,amount\n2026-01-01,Food,0.10\n2026-01-02,Food,0.20\n"
total, groups = summarize_csv(sample)
assert str(total) == "0.30" and groups == {"Food": total}
assert summarize_csv("date,category,amount\n")[0] == 0
print("Exercise 2: checks passed")


**Why this works:** Parsing belongs at the boundary; downstream calculations can assume valid typed values.


## Exercise 3: JSON contract

Serialize a total Decimal and currency into JSON strings, using keys total and currency. Preserve the decimal text.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def report_json(total, currency="USD"):
    import json
    return json.dumps({"total": str(total), "currency": currency}, sort_keys=True)


In [ ]:
import json
assert json.loads(report_json(Decimal("0.30"))) == {"total": "0.30", "currency": "USD"}
print("Exercise 3: checks passed")


**Why this works:** A JSON string prevents a consumer from assuming exact decimal arithmetic from a binary float.


## Independent transfer

Build the expense CLI in Stage 01: total, category/month summaries, mean, maximum, and bad-row diagnostics. Use a temporary directory for file tests.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [01c interface and acceptance cases](../assignments/01c.md).
2. Copy the [blank starter](../assignments/starters/01c.py) to `work/assignments/01c.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 01c --solution work/assignments/01c.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/01c.md#01c-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 01c --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why not split CSV manually?

   Commas and newlines may occur inside quoted fields.

2. What is a useful error message?

   One that identifies the record and invalid field without exposing unrelated sensitive data.


## Reading and review

- [Primary reference 1](https://docs.python.org/3/library/csv.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
